# WeedDet v5b — Phase 2: Neck + Head Fine-Tuning
**AgriNav | Benny Merryman-Smith**

Loads `weeddet_v5_best.pth`, freezes the Det-ResNet-50 backbone entirely,
and fine-tunes only the eFPN neck + ERetinaHead for 8 epochs.

**Run AFTER weeddet_trainingV5.ipynb has completed.**

| Cell | Purpose |
|------|---------|
| 0 | Mount Drive + load weeddet_Latest |
| 1 | Rebuild flat dataset (skip if already done in v5 session) |
| 2 | Load v5 checkpoint, freeze backbone, verify trainable params |
| 3 | Fine-tune neck + head (8 epochs, lower LR) |
| 4 | Plot Phase 2 loss curves |


In [ ]:
from google.colab import drive
import sys, os, torch

drive.mount('/content/drive')
SCRIPT_DIR = '/content/drive/MyDrive/weeddet_v2_checkpoints'
sys.path.insert(0, SCRIPT_DIR)

script_file = os.path.join(SCRIPT_DIR, 'weeddet_Latest.py')
assert os.path.exists(script_file), f'Script not found: {script_file}'

for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

CKPT_DIR_V5  = '/content/drive/MyDrive/weeddet_v5_checkpoints'
CKPT_DIR_V5B = '/content/drive/MyDrive/weeddet_v5b_checkpoints'
FLAT_ROOT    = '/content/rice_flat'

os.makedirs(CKPT_DIR_V5B, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'v5 source : {CKPT_DIR_V5}')
print(f'v5b output: {CKPT_DIR_V5B}')


## Cell 1 — Rebuild Dataset (skip if FLAT_ROOT already exists from v5 session)

In [ ]:
import zipfile, glob, random
from pathlib import Path

ZIP_PATH   = '/content/drive/MyDrive/weeddet_v2_checkpoints/rice_detection_for_export.v1i.voc.zip'
EXTRACT_DIR = '/content/dataset'

# Only re-extract if needed
if not os.path.exists(FLAT_ROOT) or not Path(f'{FLAT_ROOT}/val.txt').exists():
    print('Rebuilding flat dataset...')
    if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall(EXTRACT_DIR)

    os.makedirs(f'{FLAT_ROOT}/images',      exist_ok=True)
    os.makedirs(f'{FLAT_ROOT}/annotations', exist_ok=True)

    all_stems = []
    for split in ['train', 'valid', 'val', 'test']:
        base = Path(f'{EXTRACT_DIR}/{split}')
        if not base.exists(): continue
        img_dir = base / 'images' if (base / 'images').exists() else base
        ann_dir = (base / 'annotations' if (base / 'annotations').exists() else
                   base / 'labels' if (base / 'labels').exists() else base)
        for img_p in sorted(img_dir.glob('*')):
            if img_p.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
            xml_p = ann_dir / (img_p.stem + '.xml')
            if not xml_p.exists(): continue
            dst_i = Path(FLAT_ROOT) / 'images' / img_p.name
            dst_x = Path(FLAT_ROOT) / 'annotations' / (img_p.stem + '.xml')
            if not dst_i.exists(): os.symlink(img_p.resolve(), dst_i)
            if not dst_x.exists(): os.symlink(xml_p.resolve(), dst_x)
            all_stems.append(img_p.stem)

    random.seed(42); random.shuffle(all_stems)
    cut = int(len(all_stems) * 0.8)
    Path(f'{FLAT_ROOT}/train.txt').write_text('\n'.join(all_stems[:cut]))
    Path(f'{FLAT_ROOT}/val.txt').write_text('\n'.join(all_stems[cut:]))
    print(f'Train: {cut}  Val: {len(all_stems)-cut}')
else:
    n_train = len(Path(f'{FLAT_ROOT}/train.txt').read_text().splitlines())
    n_val   = len(Path(f'{FLAT_ROOT}/val.txt').read_text().splitlines())
    print(f'Dataset already ready — Train: {n_train}  Val: {n_val}')


## Cell 2 — Load v5 Checkpoint, Freeze Backbone
Freezes `model.backbone` (Det-ResNet-50) entirely.  
Only `model.fpn` (eFPN) and `model.head` (ERetinaHead) remain trainable.


In [ ]:
import torch.nn as nn

V5_CKPT = f'{CKPT_DIR_V5}/weeddet_v5_best.pth'
assert os.path.exists(V5_CKPT), (
    f'v5 checkpoint not found: {V5_CKPT}\n'
    'Run weeddet_trainingV5.ipynb first.'
)

ckpt = torch.load(V5_CKPT, map_location=device)
model = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded v5 epoch {ckpt["epoch"]}  val_loss={ckpt["loss"]:.4f}')

# ── Freeze backbone ────────────────────────────────────────────────────────────
for param in model.backbone.parameters():
    param.requires_grad = False

# Also freeze all BN everywhere (backbone + neck + head)
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        for p in m.parameters(): p.requires_grad = False

# ── Verify ────────────────────────────────────────────────────────────────────
total   = sum(p.numel() for p in model.parameters()) / 1e6
frozen  = sum(p.numel() for p in model.parameters() if not p.requires_grad) / 1e6
trainable = total - frozen

print(f'\nParameter breakdown:')
print(f'  Total     : {total:.2f}M')
print(f'  Frozen    : {frozen:.2f}M  (backbone + all BN)')
print(f'  Trainable : {trainable:.2f}M  (eFPN neck + ERetinaHead)')

# Check which modules are trainable
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters() if p.requires_grad) / 1e6
    status = 'TRAINABLE' if params > 0 else 'frozen'
    print(f'  {name:<12s}: {params:.2f}M  [{status}]')


## Cell 3 — Fine-Tune Neck + Head (8 Epochs)
Lower learning rate (0.0001) since backbone features are fixed.
Saves `weeddet_v5b_best.pth` when val loss improves.


In [ ]:
from torch.utils.data import DataLoader

NUM_EPOCHS  = 8
BASE_LR     = 0.0001    # Lower LR — backbone is frozen, features already good
MIN_LR      = 0.000005
GRAD_CLIP   = 0.3
BATCH_SIZE  = 2
SAVE_EVERY  = 4

def collate(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch)) if batch else ([], [])

train_loader = DataLoader(
    wd.WeedDataset(FLAT_ROOT, 'train', img_size=512, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
    num_workers=2, pin_memory=True)
val_loader = DataLoader(
    wd.WeedDataset(FLAT_ROOT, 'val', img_size=512, augment=False),
    batch_size=1, shuffle=False, collate_fn=collate, num_workers=2)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=BASE_LR, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS * len(train_loader), eta_min=MIN_LR)

train_losses_p2, val_losses_p2 = [], []
best_val_loss = ckpt['loss']   # must beat v5 best to save
print(f'Starting from v5 best val loss: {best_val_loss:.4f}')
print(f'Training {NUM_EPOCHS} epochs | {len(train_loader)} batches/epoch\n')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()

    epoch_loss, n_batches = 0.0, 0
    for imgs, targets in train_loader:
        if not imgs: continue
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step(); scheduler.step()
        epoch_loss += loss.item(); n_batches += 1
    avg_train = epoch_loss / max(n_batches, 1)
    train_losses_p2.append(avg_train)

    model.eval(); val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            if not imgs: continue
            imgs    = [i.to(device) for i in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            model.train()
            for m in model.modules():
                if isinstance(m, nn.BatchNorm2d): m.eval()
            val_loss += sum(model(imgs, targets).values()).item()
            model.eval(); n_val += 1
    avg_val = val_loss / max(n_val, 1)
    val_losses_p2.append(avg_val)

    star = ''
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({'epoch': epoch, 'loss': avg_val, 'phase': 'v5b',
                    'state_dict': model.state_dict(),
                    'optimizer': optimizer.state_dict()},
                   f'{CKPT_DIR_V5B}/weeddet_v5b_best.pth')
        star = '  ★ Best'

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}  '
          f'best={best_val_loss:.4f}  lr={optimizer.param_groups[0]["lr"]:.6f}{star}')

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch': epoch, 'loss': avg_val, 'state_dict': model.state_dict()},
                   f'{CKPT_DIR_V5B}/weeddet_v5b_epoch{epoch}.pth')

print(f'\nPhase 2 complete. Best val loss: {best_val_loss:.4f}')
print(f'Checkpoint: {CKPT_DIR_V5B}/weeddet_v5b_best.pth')


## Cell 4 — Phase 2 Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
epochs_p2 = list(range(1, len(train_losses_p2) + 1))
ax.plot(epochs_p2, train_losses_p2, marker='o', label='Train loss (P2)', color='steelblue')
ax.plot(epochs_p2, val_losses_p2,   marker='s', label='Val loss (P2)',   color='tomato')
if val_losses_p2:
    best_ep = val_losses_p2.index(min(val_losses_p2)) + 1
    ax.axvline(best_ep, color='tomato', linestyle='--', alpha=0.5,
               label=f'Best val (epoch {best_ep}, loss={min(val_losses_p2):.4f})')
ax.set_xlabel('Phase 2 Epoch'); ax.set_ylabel('Loss')
ax.set_title('WeedDet v5b — Neck+Head Fine-Tuning Loss')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(f'{CKPT_DIR_V5B}/loss_curve_p2.png', dpi=150); plt.show()
